In [35]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder , StandardScaler , MinMaxScaler
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.feature_selection import SelectKBest , chi2
from sklearn.tree import DecisionTreeClassifier
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error , mean_squared_error , r2_score , accuracy_score



In [3]:
df = pd.read_csv('Vehicle_emissions.csv')

In [5]:
df.head()

,Model_Year,Make,Model,Vehicle_Class,Engine_Size,Cylinders,Transmission,Fuel_Consumption_in_City(L/100 km),Fuel_Consumption_in_City_Hwy(L/100 km),Fuel_Consumption_comb(L/100km),CO2_Emissions,Smog_Level
0,2021,Acura,ILX,Compact,2.4,4,AM8,9.9,7.0,8.6,199,3
1,2021,Acura,NSX,Two-seater,3.5,6,AM9,11.1,10.8,11.0,256,3
2,2021,Acura,RDX SH-AWD,SUV: Small,2.0,4,AS10,11.0,8.6,9.9,232,6
3,2021,Acura,RDX SH-AWD A-SPEC,SUV: Small,2.0,4,AS10,11.3,9.1,10.3,242,6
4,2021,Acura,TLX SH-AWD,Compact,2.0,4,AS10,11.2,8.0,9.8,230,7


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 935 entries, 0 to 934
Data columns (total 12 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Model_Year                              935 non-null    int64  
 1   Make                                    935 non-null    object 
 2   Model                                   935 non-null    object 
 3   Vehicle_Class                           935 non-null    object 
 4   Engine_Size                             935 non-null    float64
 5   Cylinders                               935 non-null    int64  
 6   Transmission                            935 non-null    object 
 7   Fuel_Consumption_in_City(L/100 km)      935 non-null    float64
 8   Fuel_Consumption_in_City_Hwy(L/100 km)  935 non-null    float64
 9   Fuel_Consumption_comb(L/100km)          935 non-null    float64
 10  CO2_Emissions                           935 non-null    int64 

# create features and target variable

In [36]:
X = df.drop('CO2_Emissions', axis=1)
y = df['CO2_Emissions']

# split cat and num features

In [48]:
numerical_cols = ["Model_Year","Engine_Size","Cylinders","Fuel_Consumption_in_City(L/100 km)","Fuel_Consumption_in_City_Hwy(L/100 km)","Fuel_Consumption_comb(L/100km)","Smog_Level"]
categorical_cols = ["Make","Model","Vehicle_Class","Transmission"]


# start the Pipeline w/ Encoding

In [49]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),# to handle missing values ,replace it with mean
    ('scaler', StandardScaler())# to standardize the numerical features like mean = 0 and std = 1
])

In [50]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), # to handle missing values ,replace it with most frequent value
    ('encoder', OneHotEncoder(handle_unknown='ignore')) # to convert categorical features into numerical format
])

# join the pipeline together

In [51]:
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])


In [52]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),  # Ensure 'preprocessor' is defined and executed in a previous cell
    ('model', RandomForestClassifier())  # RandomForestClassifier is already imported
])

In [53]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# train the model

In [58]:
pipeline.fit(X_train, y_train)
predict_train = pipeline.predict(X_train)


In [55]:
encoded_cols = pipeline.named_steps['preprocessor'].named_transformers_['cat']['encoder'].get_feature_names_out(categorical_cols)


In [56]:
encoded_cols


array(['Make_Acura', 'Make_Alfa Romeo', 'Make_Aston Martin', 'Make_Audi',
       'Make_BMW', 'Make_Bentley', 'Make_Bugatti', 'Make_Buick',
       'Make_Cadillac', 'Make_Chevrolet', 'Make_Chrysler', 'Make_Dodge',
       'Make_FIAT', 'Make_Ford', 'Make_GMC', 'Make_Genesis', 'Make_Honda',
       'Make_Hyundai', 'Make_Infiniti', 'Make_Jaguar', 'Make_Jeep',
       'Make_Kia', 'Make_Lamborghini', 'Make_Lexus', 'Make_Lincoln',
       'Make_MINI', 'Make_Maserati', 'Make_Mazda', 'Make_Mercedes-Benz',
       'Make_Mitsubishi', 'Make_Nissan', 'Make_Porsche', 'Make_Ram',
       'Make_Rolls-Royce', 'Make_Subaru', 'Make_Toyota',
       'Make_Volkswagen', 'Make_Volvo', 'Model_1500',
       'Model_1500 4X4 EcoDiesel', 'Model_1500 4X4 TRX',
       'Model_1500 4X4 eTorque', 'Model_1500 Classic',
       'Model_1500 Classic 4X4', 'Model_1500 EcoDiesel',
       'Model_1500 HFE EcoDiesel', 'Model_1500 HFE eTorque',
       'Model_1500 eTorque', 'Model_228i xDrive Gran Coupe',
       'Model_230i xDrive Coupe'

In [59]:
predict_test = pipeline.predict(X_test)  # Generate predictions for the test set
mse = mean_squared_error(y_test, predict_test)  # Calculate MSE using test set predictions


In [60]:
mse

167.7433155080214

In [61]:
rmse = np.sqrt(mse)  # Calculate RMSE

In [62]:
rmse

np.float64(12.951575792467162)

In [63]:
r2 = r2_score(y_test, predict_test)  # Calculate R-squared using test set predictions
r2

0.9583579855497252

In [65]:
mae = mean_absolute_error(y_test, predict_test)  # Calculate MAE using test set predictions
mae

6.010695187165775

In [66]:
joblib.dump(pipeline, 'vehicle_emissions_model.pkl')

['vehicle_emissions_model.pkl']